# Proyecto Etapa 2

### PARTICIONAMIENTO Y MUESTREO EN BIG DATA CON PYSPARK

##### Dataset: Microsoft_GUIDE_Train.csv

##### Objetivo: Construir particiones representativas del dataset utilizando muestreo estratificado multivariable.

### Descripción del dataset

El dataset Microsoft_GUIDE_Train.csv contiene eventos, alertas e incidentes de ciberseguridad provenientes de múltiples organizaciones.

Características principales:
- 9.5 millones de registros
- 45 variables
- Datos reales de operaciones SOC
- Clasificación supervisada de incidentes

Variable objetivo:
- IncidentGrade

Variables de caracterización:
- IncidentGrade
- Category
- EntityType
- EvidenceRoles

### Cargando el entorno de PySpark en Goolge Colab

In [2]:
!apt-get install openjdk-8-jdk-headless -qq > /dev/null
!wget -q https://dlcdn.apache.org/spark/spark-3.5.8/spark-3.5.8-bin-hadoop3.tgz
!tar -xzf spark-3.5.8-bin-hadoop3.tgz
!pip install -q findspark

In [3]:
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-8-openjdk-amd64"
os.environ["SPARK_HOME"] = "/content/spark-3.5.8-bin-hadoop3"

In [4]:
!ls

sample_data  spark-3.5.8-bin-hadoop3  spark-3.5.8-bin-hadoop3.tgz


In [5]:
import findspark
findspark.init()
from pyspark import SparkContext, SparkConf, SQLContext
from pyspark.sql import SparkSession
#spark = SparkSession.builder.master("local[*]").getOrCreate()
spark = SparkSession.builder \
                    .master("local[*]") \
                    .appName("BigData_Particionamiento") \
                    .getOrCreate()
spark.conf.set("spark.sql.repl.eagerEval.enabled", True) # Property used to format output tables better
spark

### Conectando con una base de datos desde Google Drive

In [6]:
import os
from google.colab import drive
drive.mount('/content/drive')
file_path = "/content/drive/MyDrive/Colab Notebooks/MNA/BigData/ProyectoEtapa1/Microsoft_GUIDE_Train.csv"

Mounted at /content/drive


### Creando un Dataframe a partir de un archivo "csv" de entrada

In [7]:
df = spark.read.csv(file_path, header=True, sep=",", inferSchema=True)
df.show(10)

+-------------+-----+----------+-------+-------------------+----------+----------+------------------+---------------+--------------+-------------+--------------+-----------+------------+--------+------+---------+------+----------+----------+---------------+-----------+----------+----------------+--------------+-----------+-----------------+-----------------+-------------+---------------+------------------+------------+--------+----------+--------------+------------+-----+--------+---------+-----------------+--------------+-----------+-----------+-----+-----+
|           Id|OrgId|IncidentId|AlertId|          Timestamp|DetectorId|AlertTitle|          Category|MitreTechniques| IncidentGrade|ActionGrouped|ActionGranular| EntityType|EvidenceRole|DeviceId|Sha256|IpAddress|   Url|AccountSid|AccountUpn|AccountObjectId|AccountName|DeviceName|NetworkMessageId|EmailClusterId|RegistryKey|RegistryValueName|RegistryValueData|ApplicationId|ApplicationName|OAuthApplicationId|ThreatFamily|FileName|Fold

### Preprocesamiento Básico

In [8]:
# Eliminar registros sin variable objetivo
from pyspark.sql.functions import *
from pyspark.sql.types import *

df = df.filter(col("IncidentGrade").isNotNull())

In [9]:
# Eliminar duplicados
df = df.dropDuplicates()

In [10]:
# Cantidad de registros
print("Cantidad de registros:")
total = df.count()
print(total)

Cantidad de registros:
9442956


### Construcción de variables agrupadas

In [11]:
# Agrupación de Category
top_categories = [
    "InitialAccess",
    "Exfiltration",
    "SuspiciousActivity",
    "CommandAndControl",
    "Impact",
    "CredentialAccess",
    "Execution"
]

df = df.withColumn(
    "CategoryGroup",
    when(col("Category").isin(top_categories),
         col("Category"))
    .otherwise("OtherCategory")
)

In [12]:
# Agrupación de EntityType
df = df.withColumn(
    "EntityTypeGroup",
    when(col("EntityType") == "Ip", "Ip")
    .when(col("EntityType") == "User", "User")
    .otherwise("Other")
)

### Variables utilizadas para el particionamiento

Reglas de particionamiento:
- R1 ->	IncidentGrade	-> Estratificación primaria
- R2 -> Category agrupada -> Estratificación secundaria
- R3 -> EntityType agrupada -> Estratificación terciaria
- R4 -> EvidenceRole -> Estratificación operacional


In [13]:
# Variables utilizadas para el particionamiento
partition_columns = [
    "IncidentGrade",
    "CategoryGroup",
    "EntityTypeGroup",
    "EvidenceRole"
]

### Visualización de combinaciones posibles

In [14]:
# Visualización de combinaciones posibles
partition_combinations = df.select(
    partition_columns
).distinct()

print("Número total de combinaciones:")
print(partition_combinations.count())

partition_combinations.show(20, truncate=False)

Número total de combinaciones:
115
+--------------+------------------+---------------+------------+
|IncidentGrade |CategoryGroup     |EntityTypeGroup|EvidenceRole|
+--------------+------------------+---------------+------------+
|BenignPositive|SuspiciousActivity|User           |Impacted    |
|FalsePositive |OtherCategory     |Other          |Impacted    |
|BenignPositive|SuspiciousActivity|User           |Related     |
|TruePositive  |SuspiciousActivity|Ip             |Related     |
|BenignPositive|SuspiciousActivity|Other          |Impacted    |
|TruePositive  |SuspiciousActivity|Other          |Impacted    |
|FalsePositive |Exfiltration      |User           |Impacted    |
|FalsePositive |Impact            |Ip             |Related     |
|TruePositive  |CredentialAccess  |User           |Impacted    |
|FalsePositive |InitialAccess     |Ip             |Related     |
|FalsePositive |InitialAccess     |Other          |Impacted    |
|BenignPositive|InitialAccess     |Other          |Impa

### Generación de subconjuntos de particionamiento

In [15]:
# Ejemplo 1: BenignPositive + InitialAccess + Ip + Related
partition_1 = df.filter(
    (col("IncidentGrade") == "BenignPositive") &
    (col("CategoryGroup") == "InitialAccess") &
    (col("EntityTypeGroup") == "Ip") &
    (col("EvidenceRole") == "Related")
)

total_p1 = partition_1.count()
print("Cantidad registros partición 1: ", total_p1)
porcentaje = total_p1 / total * 100
print(f"Probabilidad de Ocurrencia Empírica particion 1: {porcentaje:.6f}%")

Cantidad registros partición 1:  65116
Probabilidad de Ocurrencia Empírica particion 1: 0.689572%


In [19]:
# Ejemplo 2: TruePositive + Exfiltration + User + Impacted
partition_2 = df.filter(
    (col("IncidentGrade") == "TruePositive") &
    (col("CategoryGroup") == "Exfiltration") &
    (col("EntityTypeGroup") == "User") &
    (col("EvidenceRole") == "Impacted")
)

total_p2 = partition_2.count()
print("Cantidad registros partición 2: ", total_p2)
porcentaje = total_p2 / total * 100
print(f"Probabilidad de Ocurrencia Empírica particion 2: {porcentaje:.6f}%")

Cantidad registros partición 2:  19927
Probabilidad de Ocurrencia Empírica particion 2: 0.211025%


In [20]:
# Ejemplo 3: FalsePositive + SuspiciousActivity + Other + Related
partition_3 = df.filter(
    (col("IncidentGrade") == "FalsePositive") &
    (col("CategoryGroup") == "SuspiciousActivity") &
    (col("EntityTypeGroup") == "Other") &
    (col("EvidenceRole") == "Related")
)

total_p3 = partition_3.count()
print("Cantidad registros partición 3: ", total_p3)
porcentaje = total_p3 / total * 100
print(f"Probabilidad de Ocurrencia Empírica particion 3: {porcentaje:.6f}%")

Cantidad registros partición 3:  30055
Probabilidad de Ocurrencia Empírica particion 3: 0.318280%


### Automatización de generación de particiones

In [21]:
# Para evitar crear manualmente las ~144 combinaciones.

# Obtener todas las combinaciones posibles
combinations = partition_combinations.collect()

# Diccionario para almacenar particiones
partitions = {}

for row in combinations:

    incident = row["IncidentGrade"]
    category = row["CategoryGroup"]
    entity = row["EntityTypeGroup"]
    evidence = row["EvidenceRole"]

    partition_name = f"{incident}_{category}_{entity}_{evidence}"

    subset = df.filter(
        (col("IncidentGrade") == incident) &
        (col("CategoryGroup") == category) &
        (col("EntityTypeGroup") == entity) &
        (col("EvidenceRole") == evidence)
    )

    partitions[partition_name] = subset

print("Total particiones generadas:", len(partitions))

Total particiones generadas: 115


### Mostrar tamaños de particiones y probabilidad de ocurrencia

In [22]:
# Probabilidad de Ocurrencia Empírica: Porcentaje real P(A∩B∩C∩D)

for name, subset in list(partitions.items())[:10]:
    print("="*60)
    print("Partición:", name)
    subset_count = subset.count()
    print("Registros:", subset_count)
    porcentaje = subset_count / total * 100
    print(f"Probabilidad de Ocurrencia Empírica: {porcentaje:.6f}%")

Partición: BenignPositive_SuspiciousActivity_User_Impacted
Registros: 113250
Probabilidad de Ocurrencia Empírica: 1.199307%
Partición: FalsePositive_OtherCategory_Other_Impacted
Registros: 24162
Probabilidad de Ocurrencia Empírica: 0.255873%
Partición: BenignPositive_SuspiciousActivity_User_Related
Registros: 769
Probabilidad de Ocurrencia Empírica: 0.008144%
Partición: TruePositive_SuspiciousActivity_Ip_Related
Registros: 99328
Probabilidad de Ocurrencia Empírica: 1.051874%
Partición: BenignPositive_SuspiciousActivity_Other_Impacted
Registros: 113554
Probabilidad de Ocurrencia Empírica: 1.202526%
Partición: TruePositive_SuspiciousActivity_Other_Impacted
Registros: 87702
Probabilidad de Ocurrencia Empírica: 0.928756%
Partición: FalsePositive_Exfiltration_User_Impacted
Registros: 72397
Probabilidad de Ocurrencia Empírica: 0.766677%
Partición: FalsePositive_Impact_Ip_Related
Registros: 168349
Probabilidad de Ocurrencia Empírica: 1.782800%
Partición: TruePositive_CredentialAccess_User_Imp

### Validación estadística del particionamiento

In [23]:
# Distribución original

df.groupBy("IncidentGrade") \
  .agg(count("*").alias("count")) \
  .withColumn("percentage", col("count") / total * 100) \
  .show()

+--------------+-------+------------------+
| IncidentGrade|  count|        percentage|
+--------------+-------+------------------+
|BenignPositive|4110748| 43.53242776943999|
| FalsePositive|2029564|21.492888455691205|
|  TruePositive|3302644|  34.9746837748688|
+--------------+-------+------------------+



In [24]:
# Distribución en la partición 1

partition_1.groupBy("IncidentGrade") \
    .count() \
    .show()

+--------------+-----+
| IncidentGrade|count|
+--------------+-----+
|BenignPositive|65116|
+--------------+-----+



### Conclusión
El proceso implementado permite:
* Dividir eficientemente el dataset original D
* Preservar la estructura estadística de la población
* Generar subconjuntos homogéneos
* Construir posteriormente muestras representativas M

El enfoque basado en PySpark es apropiado para escenarios de Big Data debido a:
* El tamaño masivo del dataset
* La alta cardinalidad
* La necesidad de procesamiento distribuido

Las particiones obtenidas servirán posteriormente para:
* Muestreo estratificado
* Entrenamiento de modelos de Machine Learning
* Detección de amenazas
* Análisis avanzado de incidentes de ciberseguridad

